# Preprocess

In [1]:
import os
import pandas as pd
# import shap
import warnings
from tqdm import tqdm
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import numpy as np
from utils import dictionaries as dictionary

import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Lambda
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.layers import Input
from sklearn.preprocessing import StandardScaler

# Suppress all warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=UserWarning, module='lightgbm')

# Set pandas display options
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [2]:
def construct_query(target_group_list):
    if isinstance(target_group_list, (list, tuple)):
        target_group_str = ', '.join([f"'{tg}'" for tg in target_group_list])
    else:
        target_group_str = f"'{target_group_list}'"

    query = f'''
        SELECT b.flight_id,
                b.target_group,
                f.advertiser,
                f.product,
                f.start_date,
                f.end_date,
                b.campaign_duration,
                b.daypart,
                f.mock_campaign,
                f.alcohol_kansspelen,
                b.total_grp,
                b.ster_grp,
                b.regio_grp,
                b.npo1_grp,
                b.npo2_grp,
                b.npo3_grp,
                b.[ster_1+]
        FROM [dbo].[tv.bereik_ster] b
        LEFT JOIN [dbo].[tv.flights_ster] f ON b.flight_id = f.flight_id
        WHERE b.ster_grp > 0 AND target_group IN ({target_group_str})

        ORDER BY start_date
    '''
    return query

def create_sql_alchemy_engine():
    """
    Creates and returns a SQLAlchemy engine for connecting to a SQL Server database.
    """
    quoted = quote_plus(
        "Driver={ODBC+Driver+17+for+SQL+Server};"
        "Server=tcp:stercampsql.database.windows.net;"
        "Database=SterCampNew;"
        "uid=stercampadmin;"
        f"pwd={os.getenv('SQLDB')}"
    )
    alchemy_con = f"mssql+pyodbc:///?odbc_connect={quoted}"
    engine = create_engine(alchemy_con)
    return engine

def query_to_df(query):
    engine = create_sql_alchemy_engine()
    df_query = pd.read_sql_query(query, engine)
    return df_query

def prep_dataframe(df_bereik):

    def calculate_total_overlap(row, period_list=None) -> int:

        total_overlap = 0
        for period_start, period_end in period_list:
            overlap_start = max(row.start_date, pd.to_datetime(period_start))
            overlap_end = min(row.end_date, pd.to_datetime(period_end))
            overlap_days = max(0, (overlap_end - overlap_start).days + 1)
            total_overlap += overlap_days

        return total_overlap


    df = df_bereik[["flight_id", "target_group", "start_date", "end_date", "advertiser", "product", "daypart", "mock_campaign", "alcohol_kansspelen",
                    "npo1_grp", "npo2_grp", "npo3_grp", "regio_grp", "ster_grp",
                    "ster_1+"]].reset_index(drop=True)

    df = (df.pivot_table(index=["flight_id", "target_group", "start_date", "end_date", 'advertiser', 'product', "mock_campaign", "alcohol_kansspelen"],
                        columns='daypart', fill_value=0.0,
                        values=['npo1_grp', 'npo2_grp', 'npo3_grp',
                                'ster_grp', 'ster_1+', 'regio_grp'], aggfunc='sum').reset_index())


    df.columns = ["_".join(col) if col[1] else col[0] for col in df.columns.to_flat_index()]

    # Convert start_date and end_date to datetime objects and calculate required values
    df['start_date'] = pd.to_datetime(df['start_date'], format="%d%m%y", errors='coerce')
    df['end_date'] = pd.to_datetime(df['end_date'], format="%d%m%y", errors='coerce')

    # Apply function to create extra columns
    df['timef_start_month'] = df['start_date'].dt.month
    df['timef_end_month'] = df['end_date'].dt.month
    df['timef_start_year'] = df['start_date'].dt.year
    df['timef_end_year'] = df['end_date'].dt.year
    df['timef_start_week'] = df['start_date'].dt.strftime("%V").astype(int)
    df['timef_end_week'] = df['end_date'].dt.strftime("%V").astype(int)
    df['timef_start_day'] = df['start_date'].dt.day
    df['timef_end_day'] = df['end_date'].dt.day
    df['timef_duration_days'] = df.apply(lambda row: (row['end_date'] - row['start_date']).days + 1, axis=1)

    df['timef_start_season'] = df['timef_start_month'].apply(lambda month: 'Winter' if month in [12, 1, 2] else
                                                    'Spring' if month in [3, 4, 5] else
                                                    'Summer' if month in [6, 7, 8] else
                                                    'Autumn')
    df['timef_end_season'] = df['timef_end_month'].apply(lambda month: 'Winter' if month in [12, 1, 2] else
                                                'Spring' if month in [3, 4, 5] else
                                                'Summer' if month in [6, 7, 8] else
                                                'Autumn')

    # Calculate total overlap with olympics and WK/EK
    df['timef_olympic_period_overlap'] = df.apply(calculate_total_overlap, axis=1, period_list=(dictionary.olympic_period_list))
    df['timef_football_period_overlap'] = df.apply(calculate_total_overlap, axis=1, period_list=(dictionary.football_wk_ek_list))

    # Create a new column 'new_data' indicating old or new
    df['new_data'] = pd.cut(
        df['start_date'],
        bins=[pd.Timestamp.min, pd.to_datetime('2023-09-01'), pd.Timestamp.max],
        labels=[0, 1])

    df = df.sort_values(by=['start_date', 'advertiser', 'product', 'end_date'])

    df = df.astype({
    'timef_start_year': 'int64',          # Convert to int64
    'timef_end_year': 'int64',            # Convert to int64
    'timef_start_week': 'int64',          # Convert to int64
    'timef_end_week': 'int64',            # Convert to int64
    'timef_start_season': 'category',     # Convert to category
    'timef_end_season': 'category',       # Convert to category
    'alcohol_kansspelen': 'bool',         # Ensure bool type
    'new_data': 'category'                # Ensure category type
    })

    df = pd.get_dummies(df, columns=['timef_start_season', 'timef_end_season', 'new_data'], drop_first=True)

    # transform NPO3 GRPS
    # pt = PowerTransformer(method='box-cox')
    # constant = 1e-6  # Small constant to shift the data
    # df['npo3_grp_daynight'] += constant
    # df['npo3_grp_primetime'] += constant
    # df['npo3_grp_total'] += constant

    # df['npo3_grp_daynight'] = pt.fit_transform(df[['npo3_grp_daynight']])
    # df['npo3_grp_primetime'] = pt.fit_transform(df[['npo3_grp_primetime']])
    # df['npo3_grp_total'] = pt.fit_transform(df[['npo3_grp_total']])

    return df

def add_zero_sample_rows(df):

    # Add n random rows that have a base value of 0 GRPs and bereik
    non_grp_columns = ['start_date', 'end_date', 'timef_duration_days', 'timef_olympic_period_overlap', 'timef_football_period_overlap',
                        'timef_start_day', 'timef_end_day', 'timef_start_week', 'timef_end_week', 'timef_start_month',
                        'timef_end_month', 'timef_start_year', 'timef_end_year', 'timef_start_season', 'timef_end_season',
                        'target_group', 'advertiser', 'product', 'mock_campaign', 'alcohol_kansspelen', 'new_data']

    sampled_df = df.sample(n=600, random_state=42).reset_index(drop=True)
    sampled_df_zeroed = sampled_df.copy()
    for column in sampled_df_zeroed.columns:
        if column not in non_grp_columns:
            sampled_df_zeroed[column] = 0

    df_combined = pd.concat([df, sampled_df_zeroed], ignore_index=True)

    return df_combined

def filter_outliers(df):

    q_low = df['ster_grp_total'].quantile(0.01)
    q_hi  = df['ster_grp_total'].quantile(0.99)
    df = df[(df['ster_grp_total'] < q_hi) & (df['ster_grp_total'] > q_low)]

    q_low = df['ster_1+_total'].quantile(0.01)
    q_hi  = df['ster_1+_total'].quantile(0.99)
    df = df[(df['ster_1+_total'] < q_hi) & (df['ster_1+_total'] > q_low)]

    return df

def fetch_and_prep_data(self, query):
    try:
        df = self.query_to_df(query)
        df = self.prep_dataframe(df)
        df = self.filter_outliers(df)
        df = self.add_zero_sample_rows(df)
    except Exception as e:
        print(f"Error executing query: {str(e)}")
        return None
    return df


In [3]:
NUMERICAL_COLS = ['npo1_grp_daynight', 'npo1_grp_primetime',
                  'npo2_grp_daynight', 'npo2_grp_primetime',
                  'npo3_grp_daynight', 'npo3_grp_primetime',
                  'regio_grp_daynight', 'regio_grp_primetime',
                  'timef_olympic_period_overlap', 'timef_football_period_overlap', 'timef_duration_days',
                  'timef_start_month', 'timef_end_month',
                  'timef_start_year', 'timef_end_year',
                  'timef_start_week', 'timef_end_week',
                #   'timef_end_day', 'timef_start_day'
                  ]

CATEGORICAL_COLS = [
                    'timef_start_season',
                    'timef_end_season'
                    ]

BOOLEAN_COLS = ['alcohol_kansspelen']

TARGET = ['ster_1+_total']

features = NUMERICAL_COLS + CATEGORICAL_COLS + BOOLEAN_COLS + TARGET

In [4]:
engine = create_sql_alchemy_engine()
query = construct_query('25-67')
df = query_to_df(query)
df_bereik = prep_dataframe(df)
df_bereik = add_zero_sample_rows(df_bereik)
df_bereik = filter_outliers(df_bereik)

df_bereik.head(2)

,flight_id,target_group,start_date,end_date,advertiser,product,mock_campaign,alcohol_kansspelen,npo1_grp_daynight,npo1_grp_primetime,npo1_grp_total,npo2_grp_daynight,npo2_grp_primetime,npo2_grp_total,npo3_grp_daynight,npo3_grp_primetime,npo3_grp_total,regio_grp_daynight,regio_grp_primetime,regio_grp_total,ster_1+_daynight,ster_1+_primetime,ster_1+_total,ster_grp_daynight,ster_grp_primetime,ster_grp_total,timef_start_month,timef_end_month,timef_start_year,timef_end_year,timef_start_week,timef_end_week,timef_start_day,timef_end_day,timef_duration_days,timef_olympic_period_overlap,timef_football_period_overlap,timef_start_season_Spring,timef_start_season_Summer,timef_start_season_Winter,timef_end_season_Spring,timef_end_season_Summer,timef_end_season_Winter,new_data_1
0,ee3d38f2-c9af-aa6e-d4b0-b55a47d46458,25-67,2019-01-01,2019-01-27,@leisure group,Belvilla,False,True,21.86,236.02,257.88,11.82,20.70,32.52,1.61,22.16,23.77,0.0,0.0,0.0,16.32,66.15,67.74,35.30,278.88,314.17,1,1,2019,2019,1,4,1,27,27,0,0,0,0,1,0,0,1,0
1,79229d34-72c9-f7af-ae46-43a4c271381a,25-67,2019-01-01,2019-01-20,Abn amro,Abn-amro,False,True,32.10,57.14,89.24,3.27,17.09,20.36,0.30,3.46,3.76,0.0,0.0,0.0,21.72,41.13,48.10,35.67,77.69,113.36,1,1,2019,2019,1,3,1,20,20,0,0,0,0,1,0,0,1,0


In [5]:
df_bereik = df_bereik[['npo1_grp_daynight', 'npo1_grp_primetime',
                  'npo2_grp_daynight', 'npo2_grp_primetime',
                  'npo3_grp_daynight', 'npo3_grp_primetime',
                  'regio_grp_daynight', 'regio_grp_primetime',
                  'timef_olympic_period_overlap', 'timef_football_period_overlap',
                  'timef_duration_days',
                  'timef_start_month', 'timef_end_month',
                  'timef_start_year', 'timef_end_year',
                  'timef_start_week', 'timef_end_week',
                  'alcohol_kansspelen',
                  'timef_start_season_Spring',
                'timef_start_season_Summer', 'timef_start_season_Winter',
                'timef_end_season_Spring', 'timef_end_season_Summer',
                'timef_end_season_Winter',
                  'ster_1+_total'
                  ]]

# Boost model

In [8]:
# Split data: Pre-08-2023 data for training, post-08-2023 data for fine-tuning
train_data = df_bereik[(df_bereik['timef_start_year'] < 2023) |
                       ((df_bereik['timef_start_year'] == 2023) & (df_bereik['timef_start_month'] <= 7))]

# Validation Data: From September 2023 up until 30 days before the latest data
val_data = df_bereik[((df_bereik['timef_start_year'] == 2023) & (df_bereik['timef_start_month'] >= 9)) |
                     (df_bereik['timef_start_year'] > 2023)]
val_data = val_data.iloc[:-30]  # Exclude the last 30 days from validation data

test_data = df_bereik.tail(30)

In [9]:
test_data.head(2)

,npo1_grp_daynight,npo1_grp_primetime,npo2_grp_daynight,npo2_grp_primetime,npo3_grp_daynight,npo3_grp_primetime,regio_grp_daynight,regio_grp_primetime,timef_olympic_period_overlap,timef_football_period_overlap,timef_duration_days,timef_start_month,timef_end_month,timef_start_year,timef_end_year,timef_start_week,timef_end_week,alcohol_kansspelen,timef_start_season_Spring,timef_start_season_Summer,timef_start_season_Winter,timef_end_season_Spring,timef_end_season_Summer,timef_end_season_Winter,ster_1+_total
58810,6.77,0.00,1.15,0.00,0.39,0.0,0.0,0.00,0,0,8,10,10,2024,2024,40,41,True,0,0,0,0,0,0,5.97
58811,22.05,7.33,3.33,2.29,0.00,3.1,0.0,0.54,0,0,8,10,10,2024,2024,40,41,True,0,0,0,0,0,0,13.09


In [10]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

# Separate features and target for each dataset
X_train = train_data.drop(columns=['ster_1+_total'])
y_train = train_data['ster_1+_total']
X_val = val_data.drop(columns=['ster_1+_total'])
y_val = val_data['ster_1+_total']
X_test = test_data.drop(columns=['ster_1+_total'])
y_test = test_data['ster_1+_total']

# Step 1: Pre-train Model on Old Data
# Initialize XGBoost model with settings suitable for initial training
pretrain_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric='mae'
)

# Fit the model on training data
pretrain_model.fit(X_train, y_train)

# Evaluate on validation data to check initial model performance
y_val_pred = pretrain_model.predict(X_val)
initial_val_mae = mean_absolute_error(y_val, y_val_pred)
initial_val_r2 = r2_score(y_val, y_val_pred)

print("Initial Validation MAE (pre-trained model):", initial_val_mae)
print("Initial Validation R2 (pre-trained model):", initial_val_r2)

Initial Validation MAE (pre-trained model): 2.8860671624227234
Initial Validation R2 (pre-trained model): 0.9205728560722175


In [11]:
import os
# Save the booster from the pre-trained model to a temporary file
pretrain_model.get_booster().save_model("temp_model.json")

# Initialize the fine-tuning model with the same parameters
finetune_model = xgb.XGBRegressor(
    n_estimators=50,  # Use fewer trees to avoid overfitting to the new data
    learning_rate=0.05,  # Lower learning rate for fine-tuning
    max_depth=5,
    random_state=42,
    eval_metric='mae'
)

# Load the pre-trained parameters into the fine-tune model from the saved file
finetune_model.load_model("temp_model.json")

# Fine-tune on the new data (validation set minus the last 30 days)
finetune_model.fit(X_val, y_val)

# Delete the temporary file after loading
os.remove("temp_model.json")

# Evaluate the fine-tuned model on the test set (last 30 days)
y_test_pred = finetune_model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Test MAE (fine-tuned model):", test_mae)
print("Test R2 (fine-tuned model):", test_r2)

Test MAE (fine-tuned model): 3.049398183822632
Test R2 (fine-tuned model): 0.8310268691580233


In [12]:
# Prepare the DataFrame for test data evaluation
test_results = X_test.copy()  # Copy the feature columns from test data
test_results['actual_reach_1_plus'] = y_test  # Add actual target values
test_results['predicted_reach_1_plus'] = y_test_pred  # Add predictions from fine-tuned model
test_results['residuals'] = test_results['actual_reach_1_plus'] - test_results['predicted_reach_1_plus']  # Calculate residuals

# Save the results to a CSV file
test_results.to_csv("test_results_with_predictions_and_residuals.csv", index=False)

print("Test results saved to 'test_results_with_predictions_and_residuals.csv'")

Test results saved to 'test_results_with_predictions_and_residuals.csv'


# Boost met CV

In [13]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
import os

# Initialize variables to track the best parameters
best_mae = float("inf")
best_model = None

# Separate features and target
X_train = train_data.drop(columns=['ster_1+_total'])
y_train = train_data['ster_1+_total']
X_val = val_data.drop(columns=['ster_1+_total'])
y_val = val_data['ster_1+_total']
X_test = test_data.drop(columns=['ster_1+_total'])
y_test = test_data['ster_1+_total']

# Initialize time-series cross-validation
tscv = TimeSeriesSplit(n_splits=5)

# Cross-validation with pre-training on the old data
for train_idx, val_idx in tscv.split(X_train):
    X_cv_train, X_cv_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_cv_train, y_cv_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Initialize and fit XGBoost model
    model = xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.07,
        max_depth=7,
        random_state=42,
        eval_metric='rmse',
        model__drop_rate = 0.2
    )
    model.fit(X_cv_train, y_cv_train, eval_set=[(X_cv_val, y_cv_val)], verbose=False)

    # Validate and store the best model
    y_cv_pred = model.predict(X_cv_val)
    cv_mae = mean_absolute_error(y_cv_val, y_cv_pred)

    if cv_mae < best_mae:
        best_mae = cv_mae
        best_model = model

# Save the best pre-trained model
best_model.get_booster().save_model("temp_best_model.json")

# Fine-tune on new data (post-08-2023) using the best pre-trained model
finetune_model = xgb.XGBRegressor(
    n_estimators=50,  # Fewer trees for fine-tuning
    learning_rate=0.05,  # Lower learning rate
    max_depth=5,
    random_state=42,
    eval_metric='rmse',
    model__drop_rate = 0.2
)

# Load pre-trained weights into the fine-tune model
finetune_model.load_model("temp_best_model.json")

# Fine-tune on the validation data
finetune_model.fit(X_val, y_val, eval_set=[(X_val, y_val)], verbose=False)

# Clean up temporary model file
os.remove("temp_best_model.json")

# Evaluate the fine-tuned model on the final test set (last 30 days)
y_test_pred = finetune_model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Best MAE from Cross-Validation on Old Data:", best_mae)
print("Test MAE (fine-tuned model):", test_mae)
print("Test R2 (fine-tuned model):", test_r2)

Best MAE from Cross-Validation on Old Data: 2.7452588436154484
Test MAE (fine-tuned model): 3.036964988708496
Test R2 (fine-tuned model): 0.8281005882632815


In [14]:
# Prepare the DataFrame for test data evaluation
test_results = X_test.copy()  # Copy the feature columns from test data
test_results['actual_reach_1_plus'] = y_test  # Add actual target values
test_results['predicted_reach_1_plus'] = y_test_pred  # Add predictions from fine-tuned model
test_results['residuals'] = test_results['actual_reach_1_plus'] - test_results['predicted_reach_1_plus']  # Calculate residuals

# Save the results to a CSV file
test_results.to_csv("test_results_with_predictions_and_residuals.csv", index=False)

print("Test results saved to 'test_results_with_predictions_and_residuals.csv'")

Test results saved to 'test_results_with_predictions_and_residuals.csv'


# Neural network

In [26]:
df_bereik.head()

,npo1_grp_daynight,npo1_grp_primetime,npo2_grp_daynight,npo2_grp_primetime,npo3_grp_daynight,npo3_grp_primetime,regio_grp_daynight,regio_grp_primetime,timef_olympic_period_overlap,timef_football_period_overlap,timef_duration_days,timef_start_month,timef_end_month,timef_start_year,timef_end_year,timef_start_week,timef_end_week,alcohol_kansspelen,timef_start_season_Spring,timef_start_season_Summer,timef_start_season_Winter,timef_end_season_Spring,timef_end_season_Summer,timef_end_season_Winter,ster_1+_total
0,21.86,236.02,11.82,20.70,1.61,22.16,0.0,0.0,0,0,27,1,1,2019,2019,1,4,True,0,0,1,0,0,1,67.74
1,32.10,57.14,3.27,17.09,0.30,3.46,0.0,0.0,0,0,20,1,1,2019,2019,1,3,True,0,0,1,0,0,1,48.10
2,65.86,25.11,1.10,3.51,0.55,3.68,0.0,0.0,0,0,23,1,1,2019,2019,1,4,True,0,0,1,0,0,1,35.36
3,148.95,160.00,19.77,71.68,2.18,29.96,0.0,0.0,0,0,20,1,1,2019,2019,1,3,True,0,0,1,0,0,1,60.04
5,134.59,395.60,29.30,36.65,6.20,20.31,0.0,0.0,0,0,28,1,1,2019,2019,1,5,True,0,0,1,0,0,1,63.94


In [27]:
df_scaled = df_bereik.copy()

numerical_cols = [
    'npo1_grp_daynight', 'npo1_grp_primetime', 'npo2_grp_daynight',
    'npo2_grp_primetime', 'npo3_grp_daynight', 'npo3_grp_primetime',
    'regio_grp_daynight', 'regio_grp_primetime', 'timef_olympic_period_overlap',
    'timef_football_period_overlap', 'timef_duration_days', 'ster_1+_total'
]

scaler = StandardScaler()
df_scaled[numerical_cols] = scaler.fit_transform(df_scaled[numerical_cols])

In [28]:
df_scaled = df_scaled.astype({
    'timef_start_month': np.int32,
    'timef_end_month': np.int32,
    'timef_start_year': np.int32,
    'timef_end_year': np.int32,
    'timef_start_week': np.int32,
    'timef_end_week': np.int32,
    'timef_start_season_Spring': np.int32,
    'timef_start_season_Summer': np.int32,
    'timef_start_season_Winter': np.int32,
    'timef_end_season_Spring': np.int32,
    'timef_end_season_Summer': np.int32,
    'timef_end_season_Winter': np.int32,
    'alcohol_kansspelen': np.int32  # Converting bool to int (0 or 1)
})

df_scaled = df_scaled.astype({col: np.float32 for col in train_data.select_dtypes(include='float64').columns})

In [29]:
# Split data: Pre-08-2023 data for training, post-08-2023 data for fine-tuning
train_data = df_scaled[(df_scaled['timef_start_year'] < 2023) |
                       ((df_scaled['timef_start_year'] == 2023) & (df_scaled['timef_start_month'] <= 7))]

# Validation Data: From September 2023 up until 30 days before the latest data
val_data = df_scaled[((df_scaled['timef_start_year'] == 2023) & (df_scaled['timef_start_month'] >= 9)) |
                     (df_scaled['timef_start_year'] > 2023)]
val_data = val_data.iloc[:-30]  # Exclude the last 30 days from validation data

test_data = df_scaled.tail(30)

In [30]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import matplotlib.pyplot as plt

# Define an improved neural network architecture
def create_improved_model():
    model = Sequential()
    model.add(Dense(128, input_shape=(X_train.shape[1],), activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))

    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))

    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))

    model.add(Dense(32, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))

    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='linear'))  # Output layer for regression
    return model

# Initialize the model
model = create_improved_model()
model.compile(optimizer=Adam(learning_rate=0.01), loss='mean_squared_error')

# Set up EarlyStopping and ReduceLROnPlateau callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1, min_lr=1e-6)

# Train the model on the old data with callbacks
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Fine-tune the model on the new data with a lower learning rate
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
fine_tune_history = model.fit(
    X_val, y_val,
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Step 2: Fine-tune on the new data
# Compile the model with a lower learning rate for fine-tuning
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

# Fine-tune the model on the new data (validation set minus the last 30 days)
model.fit(X_val, y_val, epochs=20, batch_size=16, verbose=1)

# Step 3: Evaluate the Fine-Tuned Model on the Test Set (last 30 days)
y_test_pred = model.predict(X_test).flatten()
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Test MAE (fine-tuned model):", test_mae)
print("Test R2 (fine-tuned model):", test_r2)
print("Test MSE (fine-tuned model):", test_mse)


Epoch 1/100
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.4280 - val_loss: 11.0369 - learning_rate: 0.0100
Epoch 2/100
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.1756 - val_loss: 44.2919 - learning_rate: 0.0100
Epoch 3/100
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.1630 - val_loss: 45.1630 - learning_rate: 0.0100
Epoch 4/100
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.1547 - val_loss: 12.0721 - learning_rate: 0.0100
Epoch 5/100
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1609 - val_loss: 815.1461 - learning_rate: 0.0100
Epoch 6/100
1349/1365 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1571
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.1571 - val_loss: 728.0828 - learning_rate: 0.0100
Epoch 7/100
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.1465 - val_loss: 181.0908 - learning_rate: 0.0050
Epoch 8/100
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.1

In [25]:

# Save the test results to a DataFrame with residuals
test_results = pd.DataFrame(X_test, columns=test_data.drop(columns=['ster_1+_total']).columns)
test_results['actual_reach_1_plus'] = y_test
test_results['predicted_reach_1_plus'] = y_test_pred
test_results['residuals'] = test_results['actual_reach_1_plus'] - test_results['predicted_reach_1_plus']
test_results.to_csv("neural_network_with_predictions_and_residuals.csv", index=False)

# Display the DataFrame for review
import ace_tools as tools; tools.display_dataframe_to_user(name="Neural Network Test Data Predictions and Residuals", dataframe=test_results)

ModuleNotFoundError: No module named 'ace_tools'

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit

# Step 1: Initialize the model
model = build_improved_transfer_model(X_train.shape[1])

# Define callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# # Example of TimeSeriesSplit with 3 folds
# tscv = TimeSeriesSplit(n_splits=3)
# for train_index, val_index in tscv.split(X_train):
#     X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
#     y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]
#     # Train the model on each fold
#     pretrain_history = model.fit(X_train_fold, y_train_fold, validation_data=(X_val_fold, y_val_fold), epochs=50, batch_size=32, callbacks=[reduce_lr, early_stopping])


# # Step 2: Pre-train on old data
pretrain_history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val), callbacks=[reduce_lr, early_stopping])

# Step 3: Fine-tune on new data
finetune_history = model.fit(X_val, y_val, epochs=30, batch_size=16, verbose=1, validation_data=(X_val, y_val), callbacks=[reduce_lr, early_stopping])

# Step 4: Test the Model on the Last 30 Days
test_loss = model.evaluate(X_test, y_test, verbose=1)
print("Test Loss on Last 30 Days:", test_loss)

# Plotting the training and validation loss
plt.figure(figsize=(12, 6))

# Pre-training loss
plt.plot(pretrain_history.history['loss'], label='Pre-train Loss')
plt.plot(pretrain_history.history['val_loss'], label='Pre-train Validation Loss')

# Fine-tuning loss
fine_tune_epochs = range(len(pretrain_history.history['loss']),
                         len(pretrain_history.history['loss']) + len(finetune_history.history['loss']))
plt.plot(fine_tune_epochs, finetune_history.history['loss'], label='Fine-tune Loss')
plt.plot(fine_tune_epochs, finetune_history.history['val_loss'], label='Fine-tune Validation Loss')

# Labels and legend
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Mean Squared Error (MSE)')
plt.legend()
plt.grid(True)
plt.show()


In [50]:
def preprocess_for_prediction(df, scaler, columns):
    # Convert categorical columns to categories
    df['timef_start_season'] = df['timef_start_season'].astype('category')
    df['timef_end_season'] = df['timef_end_season'].astype('category')

    # Apply one-hot encoding and align with the training columns
    df = pd.get_dummies(df, columns=['timef_start_season', 'timef_end_season'], drop_first=True)
    df = df.reindex(columns=columns, fill_value=0)  # Ensure same dummy columns as in training

    # Scale numerical columns using the fitted scaler
    numerical_cols = [
        'npo1_grp_daynight', 'npo1_grp_primetime', 'npo2_grp_daynight',
        'npo2_grp_primetime', 'npo3_grp_daynight', 'npo3_grp_primetime',
        'regio_grp_daynight', 'regio_grp_primetime', 'timef_olympic_period_overlap',
        'timef_football_period_overlap', 'timef_duration_days'
    ]
    df[numerical_cols] = scaler.transform(df[numerical_cols])

    return df

# Extract the last month of data
last_month_data = df.tail(30)  # Assuming this is a month; adjust as necessary

# Preprocess the last month of data
columns = X_train.columns  # Ensure alignment with training features
X_last_month = preprocess_for_prediction(last_month_data.drop('ster_1+_total', axis=1), scaler, columns)

# Predict on the preprocessed data
predictions = model.predict(X_last_month)
last_month_data['predictions'] = predictions
last_month_data.to_excel('predictions_nn.xlsx', index=False)

# Display predictions
print("Predictions for the last month:", predictions)

In [ ]:
last_month_data.head()
X_last_month.head()